# Clean PG(2, 2^k) — Minimal Notebook

This notebook contains a compact, well-structured implementation of:
- A parameterized GF(2^k) factory (k = 2,3,4)
- Builders for PG(2, q) (projective points and lines)
- Collinearity test and affine-point extraction
- A configurable MRV + forward-checking coloring solver

Each major section has a code cell below it; run cells in order.

## Contents
1. Imports & utilities
2. GF(2^k) factory
3. Projective-plane builders
4. Collinearity & utilities
5. Affine extraction & sanity checks
6. Coloring solver (MRV + forward checking)
7. Example & tests
8. Notes & performance tips

In [19]:
# Imports & small utilities
from itertools import product
from functools import lru_cache
from typing import List, Tuple, Optional, Callable, Dict
import time

Point = Tuple[int,int,int]  # projective triple (x,y,z) represented as ints in 0..q-1

## GF(2^k) factory
Parameterised small-field implementation for k in {2,3,4}.
Run the following cell to define `make_field(k, irred_poly=None)` which returns a small `GF` object.

In [20]:
# GF(2^k) factory supporting k=2,3,4 with optional mul-table precompute
def make_field(k: int, irred_poly: Optional[int] = None):
    """Return a small GF object for GF(2^k).
    Representation: integers 0..(2**k-1).
    Default irreducible polys: k=2 -> 0b111 (x^2+x+1), k=3 -> 0b1011 (x^3+x+1), k=4 -> 0b10011 (x^4+x+1).
    The returned object has methods: add, mul, pow, inv, elements, precompute_mul_table (optional).
    """
    class GF:
        def __init__(self, k, irred_poly=None):
            self.k = k
            self.q = 1 << k
            if irred_poly is None:
                defaults = {2:0b111, 3:0b1011, 4:0b10011}
                if k not in defaults:
                    raise ValueError('Default polynomial not provided for this k')
                self.irred = defaults[k]
            else:
                self.irred = irred_poly
            self._mul_table = None
        def add(self, a:int, b:int) -> int:
            return a ^ b
        def mul_no_table(self, a:int, b:int) -> int:
            # multiply polynomials (bits) and reduce by irred poly
            if a == 0 or b == 0:
                return 0
            res = 0
            for i in range(self.k):
                if (a >> i) & 1:
                    res ^= (b << i)
            # reduce
            top = self.k
            while res >> top:
                shift = (res.bit_length() - 1) - top
                res ^= (self.irred << shift)
            return res & (self.q - 1)
        def mul(self, a:int, b:int) -> int:
            if self._mul_table is not None:
                return self._mul_table[a][b]
            return self.mul_no_table(a,b)
        def pow(self, a:int, e:int) -> int:
            if e == 0:
                return 1
            if a == 0:
                return 0
            res = 1
            base = a
            ee = e
            while ee:
                if ee & 1:
                    res = self.mul(res, base)
                base = self.mul(base, base)
                ee >>= 1
            return res
        def inv(self, a:int) -> int:
            if a == 0:
                raise ZeroDivisionError('0 has no inverse')
            # a^{q-2} in GF(q)
            return self.pow(a, self.q - 2)
        def elements(self):
            return list(range(self.q))
        def precompute_mul_table(self):
            table = [[0]*self.q for _ in range(self.q)]
            for i in range(self.q):
                for j in range(self.q):
                    table[i][j] = self.mul_no_table(i,j)
            self._mul_table = table
        def __repr__(self):
            return f'<GF(2^{self.k}) q={self.q} irred=0b{self.irred:b}>'
    return GF(k, irred_poly)

# Example: gf8 = make_field(3)

## Projective-plane builders
Functions to build `build_projective_points(gf)` and `build_lines_by_coeff(gf)`.
These will use the `gf` instance from the GF factory.

In [21]:
def normalize_point(pt: Point, gf) -> Point:
    x, y, z = pt
    if x != 0:
        inv = gf.inv(x)
        return (1, gf.mul(y, inv), gf.mul(z, inv))
    elif y != 0:
        inv = gf.inv(y)
        return (0, 1, gf.mul(z, inv))
    else:
        inv = gf.inv(z)
        return (0, 0, 1)

def build_projective_points(gf) -> List[Point]:
    q = gf.q
    all_pts = set()
    for x, y, z in product(range(q), repeat=3):
        if x == 0 and y == 0 and z == 0:
            continue
        all_pts.add(normalize_point((x, y, z), gf))
    return sorted(all_pts)

def lin_comb(a: int, b: int, c: int, x: int, y: int, z: int, gf) -> int:
    return gf.add(gf.add(gf.mul(a, x), gf.mul(b, y)), gf.mul(c, z))

def build_lines_by_coeff(gf) -> List[Tuple[Point, ...]]:
    q = gf.q
    coeffs = set()
    for a, b, c in product(range(q), repeat=3):
        if a == 0 and b == 0 and c == 0:
            continue
        if a != 0:
            inv = gf.inv(a); coeffs.add((1, gf.mul(b, inv), gf.mul(c, inv)))
        elif b != 0:
            inv = gf.inv(b); coeffs.add((0, 1, gf.mul(c, inv)))
        else:
            inv = gf.inv(c); coeffs.add((0, 0, 1))
    coeffs_list = sorted(coeffs)
    pts = build_projective_points(gf)
    seen = set()
    lines = []
    for (a, b, c) in coeffs_list:
        line_pts = tuple(sorted([p for p in pts if lin_comb(a, b, c, *p, gf=gf) == 0]))
        if line_pts not in seen:
            seen.add(line_pts)
            lines.append(line_pts)
    return sorted(lines)

## Collinearity & utilities
Determinant-based `det3` and `collinear` functions. Also optional caching helpers.

In [22]:
def det3(a: Point, b: Point, c: Point, gf) -> int:
    (x1, y1, z1) = a
    (x2, y2, z2) = b
    (x3, y3, z3) = c
    t1 = gf.add(gf.mul(x1, gf.mul(y2, z3)), gf.mul(y1, gf.mul(z2, x3)))
    t2 = gf.mul(z1, gf.mul(x2, y3))
    t3 = gf.add(gf.mul(x3, gf.mul(y2, z1)), gf.mul(y3, gf.mul(z2, x1)))
    t4 = gf.mul(z3, gf.mul(x2, y1))
    return gf.add(gf.add(t1, t2), gf.add(t3, t4))

def collinear(a: Point, b: Point, c: Point, gf) -> bool:
    return det3(a, b, c, gf) == 0

## Affine extraction & sanity checks
Extract affine points and run basic assertions (#points, line sizes).

In [23]:
def affine_points_from(points: List[Point]) -> List[Point]:
    return [p for p in points if p[2] != 0]

def sanity_checks(gf, points, lines):
    q = gf.q
    assert len(points) == q*q + q + 1, f'unexpected number of projective points: {len(points)} vs {q*q+q+1}'
    assert len(lines) == len(points), f'number of lines should equal number of points: {len(lines)} vs {len(points)}'
    assert all(len(line) == q+1 for line in lines), 'every line should have q+1 points'
    print('Sanity checks passed: q=', q, 'points=', len(points), 'lines=', len(lines))

## Coloring solver (MRV + forward checking)
Pure, non-destructive solver with optional progress callback and partial-write path.

In [24]:
def solve_coloring(
    affine_points: List[Point],
    num_colors: int,
    gf,
    H: Optional[Dict[int, int]] = None,
    partial_write_path: Optional[str] = None,
    progress_callback: Optional[Callable[[int, int, int], None]] = None,
    timeout: Optional[float] = None
) -> Optional[List[Optional[int]]]:
    """
    Attempt to color `affine_points` with `num_colors` so that no color class contains 3 collinear points.
    Parameters:
      affine_points: list of points (z != 0)
      num_colors: number of colors
      gf: field instance
      H: optional dict mapping point-index -> preassigned color
      partial_write_path: optional path to append best partial assignments when new max depth reached
      progress_callback: optional callable(depth, nodes_explored, max_depth) to report progress
      timeout: optional seconds after which the solver stops and returns None
    Returns: assignment list of length N or None on failure/timeout.
    """
    import sys
    N = len(affine_points)
    assign = [None] * N
    domains = [set(range(num_colors)) for _ in range(N)]
    color_class = {c: [] for c in range(num_colors)}
    change_stack = []
    nodes_explored = 0
    max_depth_reached = 0
    start_time = time.time()
    H = H or {}
    # Default progress callback: print when max depth increases
    def default_progress_callback(depth, nodes_explored, max_depth):
        print(f"[Progress] Depth: {depth}, Nodes: {nodes_explored}, Max depth: {max_depth}")
        sys.stdout.flush()
    if progress_callback is None:
        progress_callback = default_progress_callback
    # apply preassignments (H maps indices -> color)
    for idx, col in H.items():
        assign[idx] = col
        domains[idx] = {col}
        color_class[col].append(idx)
    def push(entry):
        change_stack.append(entry)
    def undo(count):
        for _ in range(count):
            typ, data = change_stack.pop()
            if typ == 'assign':
                i, old = data; assign[i] = old
            elif typ == 'domain_remove':
                i, c = data; domains[i].add(c)
            elif typ == 'color_remove':
                i, c = data; color_class[c].remove(i)
    def update_progress(depth):
        nonlocal nodes_explored, max_depth_reached
        nodes_explored += 1
        wrote = False
        if depth > max_depth_reached:
            max_depth_reached = depth
            if partial_write_path is not None:
                with open(partial_write_path, 'a') as f:
                    f.write(','.join(str(assign[i]) if assign[i] is not None else '-' for i in range(N)) + '\n')
                wrote = True
            if progress_callback is not None:
                progress_callback(depth, nodes_explored, max_depth_reached)
        return wrote
    def select_var():
        # MRV: choose unassigned variable with smallest domain
        best = None; best_size = 10**9
        for i in range(N):
            if assign[i] is None:
                s = len(domains[i])
                if s < best_size:
                    best_size = s; best = i
        return best
    def forward_check(i, color):
        # record assignment
        old = assign[i]
        assign[i] = color; push(('assign',(i,old)))
        color_class[color].append(i); push(('color_remove',(i,color)))
        # for each existing pair in color class, check third points
        for q in color_class[color]:
            if q == i: continue
            for r in range(N):
                if r == i or r == q: continue
                if assign[r] == color:
                    if collinear(affine_points[i], affine_points[q], affine_points[r], gf):
                        return False
                elif assign[r] is None:
                    if color in domains[r]:
                        if collinear(affine_points[i], affine_points[q], affine_points[r], gf):
                            domains[r].remove(color); push(('domain_remove',(r,color)))
                            if not domains[r]:
                                return False
        return True
    def backtrack(depth=0):
        if timeout is not None and (time.time() - start_time) > timeout:
            return False
        update_progress(depth)
        if all(a is not None for a in assign):
            return True
        v = select_var()
        if v is None: return True
        domain_snapshot = sorted(domains[v])
        for c in domain_snapshot:
            checkpoint = len(change_stack)
            ok = forward_check(v, c)
            if ok:
                if backtrack(depth+1):
                    return True
            undo(len(change_stack) - checkpoint)
        return False
    success = backtrack(0)
    if success: return assign
    return None


## Example & tests
Small example: build GF(8), PG(2,8), extract affine points and run solver with `num_colors=7`.

In [27]:
# Example: build GF(8), PG(2,8), extract affine points, run coloring solver with num_colors=7
gf = make_field(3)
gf.precompute_mul_table()  # speed up multiplication
points = build_projective_points(gf)
lines = build_lines_by_coeff(gf)
sanity_checks(gf, points, lines)
affine_pts = affine_points_from(points)
print(f"Affine points: {len(affine_pts)} (should be 64)")
num_colors = 7
print(f"Attempting to color affine PG(2,8) with {num_colors} colors...")
result = solve_coloring(affine_pts, num_colors, gf, timeout=30)
if result is not None:
    print("Success! Coloring:")
    print(result)
    # Save result to file in ./new directory, one line per point: (x:y:1): color, homogenized
    import os
    os.makedirs('./new', exist_ok=True)
    with open('./new/affine_pg28_coloring.txt', 'w') as f:
        for pt, color in zip(affine_pts, result):
            x, y, z = pt
            inv_z = gf.inv(z)
            hx = gf.mul(x, inv_z)
            hy = gf.mul(y, inv_z)
            f.write(f"({hx}:{hy}:1): {color}\n")
    print("Result saved to ./new/affine_pg28_coloring.txt")
else:
    print("No coloring found (timeout or impossible)")

Sanity checks passed: q= 8 points= 73 lines= 73
Affine points: 64 (should be 64)
Attempting to color affine PG(2,8) with 7 colors...
[Progress] Depth: 1, Nodes: 2, Max depth: 1
[Progress] Depth: 2, Nodes: 3, Max depth: 2
[Progress] Depth: 3, Nodes: 4, Max depth: 3
[Progress] Depth: 4, Nodes: 5, Max depth: 4
[Progress] Depth: 5, Nodes: 6, Max depth: 5
[Progress] Depth: 6, Nodes: 7, Max depth: 6
[Progress] Depth: 7, Nodes: 8, Max depth: 7
[Progress] Depth: 8, Nodes: 9, Max depth: 8
[Progress] Depth: 9, Nodes: 10, Max depth: 9
[Progress] Depth: 10, Nodes: 11, Max depth: 10
[Progress] Depth: 11, Nodes: 12, Max depth: 11
[Progress] Depth: 12, Nodes: 13, Max depth: 12
[Progress] Depth: 13, Nodes: 14, Max depth: 13
[Progress] Depth: 14, Nodes: 15, Max depth: 14
[Progress] Depth: 15, Nodes: 16, Max depth: 15
[Progress] Depth: 16, Nodes: 17, Max depth: 16
[Progress] Depth: 17, Nodes: 18, Max depth: 17
[Progress] Depth: 18, Nodes: 19, Max depth: 18
[Progress] Depth: 19, Nodes: 20, Max depth: 19


In [26]:
# Example: build GF(16), PG(2,16), extract affine points, run coloring solver with num_colors=15
gf = make_field(4)
gf.precompute_mul_table()  # speed up multiplication
points = build_projective_points(gf)
lines = build_lines_by_coeff(gf)
sanity_checks(gf, points, lines)
affine_pts = affine_points_from(points)
print(f"Affine points: {len(affine_pts)} (should be 256)")
num_colors = 15
print(f"Attempting to color affine PG(2,16) with {num_colors} colors...")
result = solve_coloring(affine_pts, num_colors, gf, timeout=60)
if result is not None:
    print("Success! Coloring:")
    print(result)
    # Save result to file in ./new directory, one line per point: (x:y:1): color, homogenized
    import os
    os.makedirs('./new', exist_ok=True)
    with open('./new/affine_pg216_coloring.txt', 'w') as f:
        for pt, color in zip(affine_pts, result):
            x, y, z = pt
            inv_z = gf.inv(z)
            hx = gf.mul(x, inv_z)
            hy = gf.mul(y, inv_z)
            f.write(f"({hx}:{hy}:1): {color}\n")
    print("Result saved to ./new/affine_pg216_coloring.txt")
else:
    print("No coloring found (timeout or impossible)")

Sanity checks passed: q= 16 points= 273 lines= 273
Affine points: 256 (should be 256)
Attempting to color affine PG(2,16) with 15 colors...
[Progress] Depth: 1, Nodes: 2, Max depth: 1
[Progress] Depth: 2, Nodes: 3, Max depth: 2
[Progress] Depth: 3, Nodes: 4, Max depth: 3
[Progress] Depth: 4, Nodes: 5, Max depth: 4
[Progress] Depth: 5, Nodes: 6, Max depth: 5
[Progress] Depth: 6, Nodes: 7, Max depth: 6
[Progress] Depth: 7, Nodes: 8, Max depth: 7
[Progress] Depth: 8, Nodes: 9, Max depth: 8
[Progress] Depth: 9, Nodes: 10, Max depth: 9
[Progress] Depth: 10, Nodes: 11, Max depth: 10
[Progress] Depth: 11, Nodes: 12, Max depth: 11
[Progress] Depth: 12, Nodes: 13, Max depth: 12
[Progress] Depth: 13, Nodes: 14, Max depth: 13
[Progress] Depth: 14, Nodes: 15, Max depth: 14
[Progress] Depth: 15, Nodes: 16, Max depth: 15
[Progress] Depth: 16, Nodes: 17, Max depth: 16
[Progress] Depth: 17, Nodes: 18, Max depth: 17
[Progress] Depth: 18, Nodes: 19, Max depth: 18
[Progress] Depth: 19, Nodes: 20, Max dep

## Notes & performance tips
- Precompute multiplication tables for k<=4 using `gf.precompute_mul_table()` to speed up hot loops.
- Cache repeated `collinear` checks when beneficial.
- Consider bitmask domains for faster domain manipulation.